In [82]:
# Step 1: Install scispacy and the model
#!pip install scispacy
#!pip install spacy
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

# Step 2: Import required libraries
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re
from collections import Counter

# Step 3: Download and load the model
try:
    nlp = spacy.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")

Named Entities:
Entity: Paracetamol, Label: ENTITY
Entity: treat, Label: ENTITY
Entity: fever, Label: ENTITY
Entity: mild, Label: ENTITY
Entity: moderate pain, Label: ENTITY


In [2]:
# Multiple ways to get and work with home directory
import os
from pathlib import Path

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
#print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi
Current working directory: /Users/naveendeevi/Datasets


In [3]:
df = pd.read_csv("Bio_Omics_Part1_V2.csv")

In [5]:
df_1 = df[['rxcui','HL_Database_medication_name','RxNav_clinical_drug_component','ingredient_name'
          ,'ingredient_tty','Medication_class','mechanism_of_action','target','drug_role'
           ,'generic_brand_drug_component','generic_drugs_exists']]

In [56]:
df_1.head(10)

,rxcui,HL_Database_medication_name,RxNav_clinical_drug_component,ingredient_name,ingredient_tty,Medication_class,mechanism_of_action,target,drug_role,generic_brand_drug_component,generic_drugs_exists
0,1049289.0,1 ml diphenhydramine hydrochloride 50 mg/ml pr...,diphenhydramine hydrochloride 50 MG/ML,diphenhydramine hydrochloride,PIN,Histamine-1 Receptor Antagonist,NaN,NaN,Augmentation,[],False
1,1719646.0,1 ml haloperidol 5 mg/ml injection,haloperidol 5 MG/ML,haloperidol,IN,Typical Antipsychotic,NaN,NaN,Unknown type,[],False
2,1013930.0,12 hr clonidine hydrochloride 0.1 mg extended ...,clonidine hydrochloride 0.1 MG,clonidine hydrochloride,PIN,Central alpha-2 Adrenergic Agonist,Adrenergic receptor alpha-2 agonist,Adrenergic receptor alpha-2,Augmentation,['clonidine hydrochloride 0.1 MG [Kapvay]'],True
3,1013937.0,12 hr clonidine hydrochloride 0.2 mg extended ...,NaN,NaN,NaN,NaN,Adrenergic receptor alpha-2 agonist,Adrenergic receptor alpha-2,Unknown type,[],False
4,1602171.0,2 ml aripiprazole 200 mg/ml prefilled syringe,aripiprazole 200 MG/ML,aripiprazole,IN,Atypical Antipsychotic,NaN,NaN,Unknown type,['aripiprazole 200 MG/ML [Abilify]'],True
5,993541.0,24 hr bupropion hydrochloride 150 mg extended ...,bupropion hydrochloride 150 MG,bupropion hydrochloride,PIN,Aminoketone,Dopamine transporter inhibitor;;; Norepinephri...,Dopamine transporter;;; Norepinephrine transpo...,Unknown type,['bupropion hydrochloride 150 MG [Wellbutrin]'],True
6,993545.0,24 hr bupropion hydrochloride 150 mg extended ...,bupropion hydrochloride 150 MG,bupropion hydrochloride,PIN,Aminoketone,Dopamine transporter inhibitor;;; Norepinephri...,Dopamine transporter;;; Norepinephrine transpo...,Unknown type,['bupropion hydrochloride 150 MG [Wellbutrin]'],True
7,993564.0,24 hr bupropion hydrochloride 300 mg extended ...,bupropion hydrochloride 300 MG,bupropion hydrochloride,PIN,Aminoketone,Dopamine transporter inhibitor;;; Norepinephri...,Dopamine transporter;;; Norepinephrine transpo...,Unknown type,['bupropion hydrochloride 300 MG [Wellbutrin]'],True
8,1874553.0,24 hr desvenlafaxine succinate 100 mg extended...,desvenlafaxine succinate 100 MG,desvenlafaxine succinate,PIN,Serotonin and Norepinephrine Reuptake Inhibitor,Serotonin transporter inhibitor;;; Norepinephr...,Serotonin transporter;;; Norepinephrine transp...,Unknown type,['desvenlafaxine succinate 100 MG [Pristiq]'],True
9,1607617.0,24 hr desvenlafaxine succinate 25 mg extended ...,desvenlafaxine succinate 25 MG,desvenlafaxine succinate,PIN,Serotonin and Norepinephrine Reuptake Inhibitor,Serotonin transporter inhibitor;;; Norepinephr...,Serotonin transporter;;; Norepinephrine transp...,Unknown type,['desvenlafaxine succinate 25 MG [Pristiq]'],True


In [69]:

drug_name = 'desvenlafaxine succinate'

#fda_url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:\"{drug_name.lower()}\""
#print(fda_url)

'''
url = "https://api.fda.gov/drug/label.json?search=openfda.brand_name:%desvenlafaxine%succinate%22"
print(url)
'''
#url = f'https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name.lower()}'

#fda_resp = requests.get(fda_url).json()
#results = fda_resp.get('results', [])
'''
if results and 'indications_and_usage' in results[0]:
    indication_large_body = results[0]['indications_and_usage'][0]
else:
    print(f"No indication info found for {drug_name}")
'''
'''
fda_resp = requests.get(url).json()
results = fda_resp.get('results', [])
if results and 'indications_and_usage' in results[0]:
    print(results[0]['indications_and_usage'][0])
else:
    print(f"No indication info found for {drug_name}")
'''

'\nfda_resp = requests.get(url).json()\nresults = fda_resp.get(\'results\', [])\nif results and \'indications_and_usage\' in results[0]:\n    print(results[0][\'indications_and_usage\'][0])\nelse:\n    print(f"No indication info found for {drug_name}")\n'

In [ ]:
#For the temporary relief of nasal decongestiom due to the commomn cold, hay fever or other upper respiratpry allergies. Temporarily relieves nasal stuffiness. Decongests nasal passages: shrinks swollen membranes. Temporarily restores freer breathing through the nose. Helps decongest sinus openings and passages; temporarily relieves sinus congestion and pressure. Promotes nasal and/or sinus drainage. temporarily relieves sinus congestion and pressure

In [76]:
def text_to_disease(text,drug):
    nlp = spacy.load("en_ner_bc5cdr_md")
    nlp.get_pipe('ner')

    #print(text)
    #print(drug)
    
    text_2 = ''.join(c.lower() if c.isalpha() else c for c in text)
    drug_2 = ''.join(c.lower() if c.isalpha() else c for c in drug)
    
    #print(text_2)
    #print(drug_2)
    
    result = nlp(text_2)
    result_2 = list(result.ents)
    result_3 = list(map(lambda x:str(x),result.ents))
    #result_4 = ''.join(c.lower() if c.isalpha() else c for c in result_3)
    #drug_2 = ''.join(c.lower() if c.isalpha() else c for c in drug)
    #print(result_4)
    result_4 = [i for i in result_3 if i != drug_2]
    counted = Counter(result_4)  # Count occurrences
    sorted_result = tuple(sorted(counted.items(), key=lambda x: x[1], reverse=True))  # Sort by count (descending)    
    return sorted_result

In [ ]:
text_to_disease()

In [110]:

drug_name = 'diazepam'
fda_url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:\"{drug_name.lower()}\""
print(fda_url)

fda_resp = requests.get(fda_url).json()
results = fda_resp.get('results', [])

if results and 'indications_and_usage' in results[0]:
    indication_large_body = results[0]['indications_and_usage'][0]
else:
    return f"No indication info found for {drug_name}"

#print(indication_large_body)

#text_to_disease()

nlp = spacy.load("en_ner_bc5cdr_md")
nlp.get_pipe('ner')

text_2 = ''.join(c.lower() if c.isalpha() else c for c in indication_large_body)

print(text_2)

result = nlp(text_2)
result_2 = list(result.ents)
result_3 = list(map(lambda x:str(x),result.ents))
#print(result_3)

result_4 = [i for i in result_3 if i != drug_name]
counted = Counter(result_4)  # Count occurrences
sorted_result = tuple(sorted(counted.items(), key=lambda x: x[1], reverse=True))  # Sort by count (descending)    
print(sorted_result)

https://api.fda.gov/drug/label.json?search=openfda.brand_name:"diazepam"
indications and usage diazepam tablets are indicated for the management of anxiety disorders or for the short-term relief of the symptoms of anxiety. anxiety or tension associated with the stress of everyday life usually does not require treatment with an anxiolytic. in acute alcohol withdrawal, diazepam tablets may be useful in the symptomatic relief of acute agitation, tremor, impending or acute delirium tremens and hallucinosis. diazepam tablets are a useful adjunct for the relief of skeletal muscle spasm due to reflex spasm to local pathology (such as inflammation of the muscles or joints, or secondary to trauma), spasticity caused by upper motor neuron disorders (such as cerebral palsy and paraplegia), athetosis, and stiff-man syndrome. oral diazepam tablets may be used adjunctively in convulsive disorders, although it has not proved useful as the sole therapy. the effectiveness of diazepam tablets in long-te

In [ ]:
"Diazepam is indicated for the management of anxiety disorders or for the short-term relief of the symptoms of anxiety. 
Anxiety or tension associated with the stress of everyday life usually does not require treatment with an anxiolytic.
In acute alcohol withdrawal, diazepam may be useful in the symptomatic relief of acute agitation, tremor
, impending or acute delirium tremens and hallucinosis.Diazepam is a useful adjunct for the relief of skeletal muscle 
spasm due to reflex spasm to local pathology (such as inflammation of the muscles or joints, or secondary to trauma), 
spasticity caused by upper motor neuron disorders (such as cerebral palsy and paraplegia), athetosis, and stiff-man syndrome.
Oral diazepam may be used adjunctively in convulsive disorders, although it has not proved useful as the sole therapy.
The effectiveness of diazepam in long-term use, that is, more than 4 months, has not been assessed by systematic clinical studies. 
The physician should periodically reassess the usefulness of the drug for the individual patient."

In [96]:
import requests

def get_indications_from_rxcui(rxcui):
    # Step 1: Get drug name
    rxcui_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
    rxcui_resp = requests.get(rxcui_url).json()
    name = rxcui_resp.get('properties', {}).get('name')
    print(name)
    if not name:
        return f"No drug name found for RxCUI {rxcui}"

    # Step 2: Get indications from FDA
    fda_url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:{name}"
    fda_resp = requests.get(fda_url).json()
    results = fda_resp.get('results', [])

    return results
    '''
    if results and 'indications_and_usage' in results[0]:
        return results[0]['indications_and_usage'][0]
    else:
        return f"No indication info found for {name}"
    '''
# Example usage
print(get_indications_from_rxcui(197590))

diazepam 2 MG Oral Tablet
[{'effective_time': '20210902', 'inactive_ingredient': ['INACTIVE INGREDIENTS Sucrose'], 'purpose': ['USES USES: Temporary Relief - Acne, Boils* * Claims based on traditional homeopathic practice, not accepted medical evidence. Not FDA evaluated.'], 'keep_out_of_reach_of_children': ['Keep this and all medication out of reach of children'], 'warnings': ['WARNINGS This product is to be used for self-limiting conditions If symptoms do not improve in 4 days, or worsen, discontinue use and seek assistance of health professional. As with any drug, if you are pregnant, or nursing a baby, seek professional advice before taking this product. Keep this and all medication out of reach of children Do not use if capseal is broken or missing. Close the cap tightly after use.'], 'questions': ['QUESTIONS OR COMMENTS www.Rxhomeo.com | 1.888.2796642 | info@rxhomeo.com Rxhomeo, Inc 3200 Commander Dr, Ste 100-W1, Carrollton, TX 75006 USA'], 'spl_product_data_elements': ['SILICEA 

In [97]:
x = get_indications_from_rxcui(197590)

diazepam 2 MG Oral Tablet


In [104]:
y = x[0]

In [106]:
y.keys()

dict_keys(['effective_time', 'inactive_ingredient', 'purpose', 'keep_out_of_reach_of_children', 'warnings', 'questions', 'spl_product_data_elements', 'openfda', 'version', 'dosage_and_administration', 'pregnancy_or_breast_feeding', 'stop_use', 'storage_and_handling', 'do_not_use', 'package_label_principal_display_panel', 'indications_and_usage', 'set_id', 'id', 'active_ingredient'])

In [107]:
y['indications_and_usage']

['INDICATIONS Condition listed above or as directed by the physician']